# DFCI analysis

In [1]:
import numpy as np
import pandas as pd
import os
import torch
from functools import partial

import contextlib
import io
import warnings
import json

from madrigal.utils import BASE_DIR
from madrigal.evaluate.predict import get_twosides_scores_wrapper

from sklearn.decomposition import PCA
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import statsmodels.api as sm
from scipy.stats import kendalltau

def sigmoid(x: np.ndarray):
    return 1 / (1 + np.exp(-x))

drug_metadata = pd.read_pickle(os.path.join(BASE_DIR, 'processed_data/drug_features/drug_metadata_ddi.pkl'))
drug_metadata['view_str'] = 1
print(drug_metadata.shape[0])

No CUDA runtime is found, using CUDA_HOME='$CUDA_HOME/'


['str', 'kg', 'cv']


21842


In [2]:
twosides_ddi_classes = pd.read_pickle(
    BASE_DIR + "processed_data/drug_combination_data/TWOSIDES/twosides_ddi_directed_label_map.pkl"
)
twosides_ckpts = [
    "all_train_seed0",
    "all_train_seed1",
    "all_train_seed2",
    "all_train_seed42",
    "all_train_seed99"
]
get_twosides_scores = partial(get_twosides_scores_wrapper, twosides_ddi_classes=twosides_ddi_classes, ckpt_list=twosides_ckpts)

outcome_mapper = json.load(open("../outcome_mapper.json", "r"))
outcome_inds_mapper = {
    ae_dfci: {"twosides": [twosides_ddi_classes.tolist().index(ae_db) for ae_db in dct["twosides"]]} 
    for ae_dfci, dct in outcome_mapper.items()
}

## Data preparation

Due to patient privacy concerns, patient-level data is not released.

In [3]:
# Build the filtered two-drug-regimen table from the raw export.
dfci_filtered_path = "./dfci/dfci_patient_data_filtered.pkl"
if not os.path.exists(dfci_filtered_path):
    dfci_patient_data = pd.read_csv("./dfci/patient_level_ae_dfci_relaxed.csv", index_col=0)
    temp = dfci_patient_data["FIRST_DRUG_REGIMEN"].value_counts()
    temp = temp[temp.index.str.contains("\+")]
    assert temp[temp.index.str.contains("SALINE")].shape[0] == 0
    assert temp[(temp.index.str.contains("/")) & (temp >= 10)].shape[0] == 0
    dfci_patient_data_filtered = dfci_patient_data[dfci_patient_data["FIRST_DRUG_REGIMEN"].str.contains("\+")]
    dfci_patient_data_filtered[["drug_name_1", "drug_name_2"]] = dfci_patient_data_filtered["FIRST_DRUG_REGIMEN"].str.replace(" HYDROCHLORIDE", "").str.replace(" MALEATE", "").str.replace(" MESYLATE", "").str.replace(" CITRATE", "").str.replace(" TRIOXIDE", "").str.replace(" TARTRATE", "").str.replace(" EMTANSINE", "").str.replace("PEGYLATEDLIPOSOMALDOXORUBICIN", "DOXORUBICIN").str.replace(" TOSYLATE", "").str.split("\+", expand=True)
    dfci_patient_data_filtered[["drug_name_1", "drug_name_2"]] = dfci_patient_data_filtered[["drug_name_1", "drug_name_2"]].apply(lambda row: [row["drug_name_1"].strip().capitalize(), row["drug_name_2"].strip().capitalize()], axis=1, result_type="expand")
    dfci_patient_data_filtered["drug_name_1"] = dfci_patient_data_filtered["drug_name_1"].str.replace("Mesna", "Coenzyme M").str.replace("Arsenic", "Arsenic trioxide").str.replace("Mirdametinib", "PD-0325901")
    dfci_patient_data_filtered["drug_name_2"] = dfci_patient_data_filtered["drug_name_2"].str.replace("Mesna", "Coenzyme M").str.replace("Arsenic", "Arsenic trioxide").str.replace("Mirdametinib", "PD-0325901")
    dfci_patient_data_filtered = dfci_patient_data_filtered[~(dfci_patient_data_filtered["drug_name_1"].str.endswith("mab") | dfci_patient_data_filtered["drug_name_2"].str.endswith("mab") | dfci_patient_data_filtered["drug_name_1"].str.endswith("vedotin") | dfci_patient_data_filtered["drug_name_2"].str.endswith("vedotin") | dfci_patient_data_filtered["drug_name_1"].isin({"Sargramostim", "Tagraxofusp-erzs", "Wee1 inhibitor zn-c3", "Sacituzumab govitecan", "Azenosertib"}) | dfci_patient_data_filtered["drug_name_2"].isin({"Sargramostim", "Tagraxofusp-erzs", "Wee1 inhibitor zn-c3", "Sacituzumab govitecan", "Azenosertib"}))]  # remove monoclonal antibodies or drugs not in our database
    dfci_patient_data_filtered = dfci_patient_data_filtered[dfci_patient_data_filtered["FIRST_DRUG_REGIMEN"].isin(dfci_patient_data_filtered["FIRST_DRUG_REGIMEN"].value_counts()[dfci_patient_data_filtered["FIRST_DRUG_REGIMEN"].value_counts() >= 10].index.values)]
    dfci_patient_data_filtered[["dbid_1", "dbid_2"]] = dfci_patient_data_filtered[["drug_name_1", "drug_name_2"]].apply(lambda row: [drug_metadata[drug_metadata["node_name"] == row["drug_name_1"]]["node_id"].values[0], drug_metadata[drug_metadata["node_name"] == row["drug_name_2"]]["node_id"].values[0]], axis=1, result_type="expand")
    dfci_patient_data_filtered[["drug_index_1", "drug_index_2"]] = dfci_patient_data_filtered[["dbid_1", "dbid_2"]].apply(lambda row: [drug_metadata[drug_metadata["node_id"] == row["dbid_1"]].index.values[0], drug_metadata[drug_metadata["node_id"] == row["dbid_2"]].index.values[0]], axis=1, result_type="expand")
    dfci_patient_data_filtered.to_pickle(dfci_filtered_path)


Index(['FIRST_DRUG_REGIMEN', 'ICD_BASED_TISSUE_TYPE', 'PALLIATIVE_INTENT',
       'RACE', 'GENDER_NM', 'AGE_AT_TREAT', 'AE_neutropenia',
       'AE_pancytopenia', 'AE_anemia', 'AE_thrombocytopenia',
       'AE_polyneuropathy', 'AE_pulmonary_embolism/deep_vein_thrombosis',
       'AE_acute_kidney_injury', 'AE_hyponatremia', 'AE_hypokalemia',
       'AE_hyperkalemia', 'AE_hypomagnesemia', 'AE_hypocalcemia',
       'AE_hypercalcemia'],
      dtype='object')

In [ ]:
dfci_patient_data_filtered = pd.read_pickle(dfci_filtered_path)
aes = [col[len("AE_"):] for col in dfci_patient_data_filtered.columns if col.startswith("AE_")]


In [ ]:
# First generate scores for each drug for each outcome
drug_names = np.unique(dfci_patient_data_filtered[["drug_name_1", "drug_name_2"]].values.flatten())
drug_inds = [drug_metadata[drug_metadata["node_name"] == drug_name].index.values[0] for drug_name in drug_names]

# Then generate scores for each drug pair for each outcome
drug_1_ind_inds = [drug_names.tolist().index(name) for name in dfci_patient_data_filtered["drug_name_1"].tolist()]
drug_2_ind_inds = [drug_names.tolist().index(name) for name in dfci_patient_data_filtered["drug_name_2"].tolist()]

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    with contextlib.redirect_stdout(io.StringIO()):
        dfci_patient_drugs_twosides_scores, _ = get_twosides_scores(
            outcome_twosides_inds=sum([[o for o in outcome_inds_mapper[ae]["twosides"] if o not in {twosides_ddi_classes.tolist().index("Adverse event"), twosides_ddi_classes.tolist().index("Adverse drug reaction")}] for ae in aes], start=[]), 
            drug_inds=drug_inds, 
            drug_group_str="dfci_patient",
        )
        
pd.DataFrame(dfci_patient_drugs_twosides_scores[700][:, drug_1_ind_inds, drug_2_ind_inds], index=sum([[o for o in outcome_mapper[ae]["twosides"] if o not in {"Adverse event", "Adverse drug reaction"}] for ae in aes], start=[]), columns=list(zip(np.array(drug_names)[drug_1_ind_inds], np.array(drug_names)[drug_2_ind_inds]))).to_pickle(BASE_DIR + "temp/dfci_patient_drugs_twosides_scores.pkl")

## Compute correlation

Map data

In [6]:
heme_drugs = {"Acalabrutinib", "Arsenic trioxide", "Azacitidine", "Bortezomib", "Busulfan", "Cytarabine", "Dasatinib", "Daunorubicin", "Decitabine", "Duvelisib", "Fludarabine", "Gilteritinib", "Ixazomib", "Lenalidomide", "Midostaurin", "Romidepsin", "Ruxolitinib", "Tretinoin", "Umbralisib", "Venetoclax", "Vincristine", "Imatinib", "Hydroxyurea", "Cyclophosphamide", "Methotrexate"}
heme_tissues = {"Myeloid", "Lymphoid", "Myeloma", "Leukemia", "Hematologic Other"}

In [7]:
dfci_data = pd.read_pickle("./dfci/dfci_patient_data_filtered.pkl")
temp = dfci_data.groupby("FIRST_DRUG_REGIMEN").agg(list)
dfci_data = dfci_data[dfci_data["FIRST_DRUG_REGIMEN"].isin(temp[temp["ICD_BASED_TISSUE_TYPE"].apply(len) >= 20].index.values)]
dfci_data = dfci_data.query("ICD_BASED_TISSUE_TYPE not in @heme_tissues")
dfci_data = dfci_data[["FIRST_DRUG_REGIMEN", "drug_name_1", "drug_name_2", "dbid_1", "dbid_2", "drug_index_1", "drug_index_2"] + [col for col in dfci_data.columns if col.startswith("AE_")]]
dfci_data = dfci_data[~dfci_data["FIRST_DRUG_REGIMEN"].str.contains("PEGYLATEDLIPOSOMALDOXORUBICIN")]  # no match
print(dfci_data.shape[0])

dfci_data = dfci_data.groupby("FIRST_DRUG_REGIMEN").agg(list)
dfci_data["drug_name_1"] = dfci_data["drug_name_1"].apply(lambda x: x[0])
dfci_data["drug_name_2"] = dfci_data["drug_name_2"].apply(lambda x: x[0])
dfci_data["dbid_1"] = dfci_data["dbid_1"].apply(lambda x: x[0])
dfci_data["dbid_2"] = dfci_data["dbid_2"].apply(lambda x: x[0])
dfci_data["drug_index_1"] = dfci_data["drug_index_1"].apply(lambda x: x[0])
dfci_data["drug_index_2"] = dfci_data["drug_index_2"].apply(lambda x: x[0])
dfci_data["num_all_patients"] = dfci_data["AE_neutropenia"].apply(lambda x: len(x))
for col in dfci_data.columns:
    if col.startswith("AE_"):
        ae = col[3:]
        dfci_data[f"num_{ae}_cases"] = dfci_data[col].apply(lambda lst: sum(lst))

dfci_data = pd.concat([
    dfci_data, 
    dfci_data.rename(
        columns={
            "drug_name_2": "drug_name_1", 
            "drug_name_1": "drug_name_2", 
            "dbid_2": "dbid_1", 
            "dbid_1": "dbid_2", 
            "drug_index_2": "drug_index_1", 
            "drug_index_1": "drug_index_2"
        }
    )
], axis=0)
dfci_data = dfci_data.query("drug_index_1 > drug_index_2").drop(columns=[col for col in dfci_data.columns if col.startswith("AE_")])
dfci_data = dfci_data.drop_duplicates(subset=["drug_index_1", "drug_index_2", "num_all_patients", "num_neutropenia_cases"])

4041


In [8]:
drug_names = np.unique(dfci_data[["drug_name_1", "drug_name_2"]].values.flatten())
drug_inds = [drug_metadata[drug_metadata["node_name"] == drug_name].index.values[0] for drug_name in drug_names]
drug_1_ind_inds = [drug_names.tolist().index(name) for name in dfci_data["drug_name_1"].tolist()]
drug_2_ind_inds = [drug_names.tolist().index(name) for name in dfci_data["drug_name_2"].tolist()]

Set a detection threshold based on exact binomial test

In [9]:
import math
def n_detect(p=0.05, power=0.95):
    return math.ceil(math.log(1-power) / math.log(1-p))
n_detect(0.05, 0.8)

32

In [10]:
detect_thres = 32

Calculate scores

In [13]:
for ae in aes:
    dfci_data[f"proportion_{ae}"] = dfci_data[f"num_{ae}_cases"] / dfci_data["num_all_patients"]
    dfci_data[["drug_name_1", "drug_name_2", "drug_index_1", "drug_index_2", f"proportion_{ae}"]].sort_values(f"proportion_{ae}", ascending=False)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        with contextlib.redirect_stdout(io.StringIO()):
            dfci_test_twosides_scores, _ = get_twosides_scores(
                outcome_twosides_inds=sum([outcome_inds_mapper[ae]["twosides"] for ae in aes], start=[]), 
                drug_inds=drug_inds, 
                drug_group_str="dfci_test",
            )
            
    torch.save(dfci_test_twosides_scores, BASE_DIR + f"temp/final_drug_regimen_ae_summary_dfci_relaxed_v2_{ae.replace('/', '_')}_scores.pt")

### Per AE correlation

#### neutropenia

In [14]:
ae = "neutropenia"
dfci_data[f"proportion_{ae}"] = dfci_data[f"num_{ae}_cases"] / dfci_data["num_all_patients"]
dfci_test_twosides_scores = torch.load(BASE_DIR + f"temp/final_drug_regimen_ae_summary_dfci_relaxed_v2_{ae}_scores.pt")

dfci_data.loc[:, f"pred_{ae}_twosides"] = pd.DataFrame(dfci_test_twosides_scores[700][:, drug_1_ind_inds, drug_2_ind_inds], index=sum([outcome_mapper[ae]["twosides"] for ae in aes], start=[]), columns=list(zip(np.array(drug_names)[drug_1_ind_inds], np.array(drug_names)[drug_2_ind_inds]))).drop_duplicates().loc["Neutropenia"].values

plot_data = dfci_data.query("drug_name_1 not in @heme_drugs and drug_name_2 not in @heme_drugs and num_all_patients >= @detect_thres")
print(plot_data.shape)
print(kendalltau(plot_data[f"proportion_{ae}"], plot_data[f"pred_{ae}_twosides"]))

(16, 34)
SignificanceResult(statistic=0.6789028582272216, pvalue=0.0003534231509884696)


#### pancytopenia

In [15]:
ae = "pancytopenia"
dfci_data[f"proportion_{ae}"] = dfci_data[f"num_{ae}_cases"] / dfci_data["num_all_patients"]
dfci_test_twosides_scores = torch.load(BASE_DIR + f"temp/final_drug_regimen_ae_summary_dfci_relaxed_v2_{ae}_scores.pt")

dfci_data.loc[:, f"pred_{ae}_twosides"] = pd.DataFrame(dfci_test_twosides_scores[700][:, drug_1_ind_inds, drug_2_ind_inds], index=sum([outcome_mapper[ae]["twosides"] for ae in aes], start=[]), columns=list(zip(np.array(drug_names)[drug_1_ind_inds], np.array(drug_names)[drug_2_ind_inds]))).drop_duplicates().loc["Pancytopenia"].values

plot_data = dfci_data.query("drug_name_1 not in @heme_drugs and drug_name_2 not in @heme_drugs and num_all_patients >= @detect_thres")
print(plot_data.shape)
print(kendalltau(plot_data[f"proportion_{ae}"], plot_data[f"pred_{ae}_twosides"]))

(16, 35)
SignificanceResult(statistic=0.4495602040932639, pvalue=0.020752734013335244)


#### anemia 

In [16]:
ae = "anemia"
dfci_data[f"proportion_{ae}"] = dfci_data[f"num_{ae}_cases"] / dfci_data["num_all_patients"]
dfci_test_twosides_scores = torch.load(BASE_DIR + f"temp/final_drug_regimen_ae_summary_dfci_relaxed_v2_{ae}_scores.pt")

dfci_data.loc[:, f"pred_{ae}_twosides"] = pd.DataFrame(dfci_test_twosides_scores[700][:, drug_1_ind_inds, drug_2_ind_inds], index=sum([outcome_mapper[ae]["twosides"] for ae in aes], start=[]), columns=list(zip(np.array(drug_names)[drug_1_ind_inds], np.array(drug_names)[drug_2_ind_inds]))).drop_duplicates().loc["Anaemia"].values

plot_data = dfci_data.query("drug_name_1 not in @heme_drugs and drug_name_2 not in @heme_drugs and num_all_patients >= @detect_thres")
print(kendalltau(plot_data[f"proportion_{ae}"], plot_data[f"pred_{ae}_twosides"]))

SignificanceResult(statistic=0.42678197846541877, pvalue=0.021534537060985532)


#### thrombocytopenia

In [17]:
ae = "thrombocytopenia"
dfci_data[f"proportion_{ae}"] = dfci_data[f"num_{ae}_cases"] / dfci_data["num_all_patients"]
dfci_test_twosides_scores = torch.load(BASE_DIR + f"temp/final_drug_regimen_ae_summary_dfci_relaxed_v2_{ae}_scores.pt")

dfci_data.loc[:, f"pred_{ae}_twosides"] = pd.DataFrame(dfci_test_twosides_scores[700][:, drug_1_ind_inds, drug_2_ind_inds], index=sum([outcome_mapper[ae]["twosides"] for ae in aes], start=[]), columns=list(zip(np.array(drug_names)[drug_1_ind_inds], np.array(drug_names)[drug_2_ind_inds]))).drop_duplicates().loc["Thrombocytopenia"].values

plot_data = dfci_data.query("drug_name_1 not in @heme_drugs and drug_name_2 not in @heme_drugs and num_all_patients >= @detect_thres")
print(kendalltau(plot_data[f"proportion_{ae}"], plot_data[f"pred_{ae}_twosides"]))

SignificanceResult(statistic=0.48741743667595394, pvalue=0.010318763168636468)


#### polyneuropathy

In [18]:
ae = "polyneuropathy"
dfci_data[f"proportion_{ae}"] = dfci_data[f"num_{ae}_cases"] / dfci_data["num_all_patients"]
dfci_test_twosides_scores = torch.load(BASE_DIR + f"temp/final_drug_regimen_ae_summary_dfci_relaxed_v2_{ae}_scores.pt")

dfci_data.loc[:, f"pred_{ae}_twosides"] = pd.DataFrame(dfci_test_twosides_scores[700][:, drug_1_ind_inds, drug_2_ind_inds], index=sum([outcome_mapper[ae]["twosides"] for ae in aes], start=[]), columns=list(zip(np.array(drug_names)[drug_1_ind_inds], np.array(drug_names)[drug_2_ind_inds]))).drop_duplicates().loc["Neuropathy peripheral"].values

plot_data = dfci_data.query("drug_name_1 not in @heme_drugs and drug_name_2 not in @heme_drugs and num_all_patients >= @detect_thres")
print(kendalltau(plot_data[f"proportion_{ae}"], plot_data[f"pred_{ae}_twosides"]))

SignificanceResult(statistic=0.4426266681379905, pvalue=0.023342202012890816)


#### pulmonary_embolism/deep_vein_thrombosis

In [19]:
ae = "pulmonary_embolism/deep_vein_thrombosis"
dfci_data[f"proportion_{ae}"] = dfci_data[f"num_{ae}_cases"] / dfci_data["num_all_patients"]
dfci_test_twosides_scores = torch.load(BASE_DIR + f"temp/final_drug_regimen_ae_summary_dfci_relaxed_v2_{ae.replace('/', '_')}_scores.pt")

dfci_data.loc[:, f"pred_{ae}_twosides"] = pd.DataFrame(dfci_test_twosides_scores[700][:, drug_1_ind_inds, drug_2_ind_inds], index=sum([outcome_mapper[ae]["twosides"] for ae in aes], start=[]), columns=list(zip(np.array(drug_names)[drug_1_ind_inds], np.array(drug_names)[drug_2_ind_inds]))).drop_duplicates().loc["Deep vein thrombosis"].values

plot_data = dfci_data.query("drug_name_1 not in @heme_drugs and drug_name_2 not in @heme_drugs and num_all_patients >= @detect_thres")
print(kendalltau(plot_data[f"proportion_{ae}"], plot_data[f"pred_{ae}_twosides"]))

SignificanceResult(statistic=0.5424508028966483, pvalue=0.003789568954129627)


#### acute_kidney_injury

In [20]:
ae = "acute_kidney_injury"
dfci_data[f"proportion_{ae}"] = dfci_data[f"num_{ae}_cases"] / dfci_data["num_all_patients"]
dfci_test_twosides_scores = torch.load(BASE_DIR + f"temp/final_drug_regimen_ae_summary_dfci_relaxed_v2_{ae}_scores.pt")
dfci_data.loc[:, f"pred_{ae}_twosides"] = pd.DataFrame(dfci_test_twosides_scores[700][:, drug_1_ind_inds, drug_2_ind_inds], index=sum([outcome_mapper[ae]["twosides"] for ae in aes], start=[]), columns=list(zip(np.array(drug_names)[drug_1_ind_inds], np.array(drug_names)[drug_2_ind_inds]))).drop_duplicates().loc["Nephropathy toxic"].values

plot_data = dfci_data.query("drug_name_1 not in @heme_drugs and drug_name_2 not in @heme_drugs and num_all_patients >= @detect_thres")
print(kendalltau(plot_data[f"proportion_{ae}"], plot_data[f"pred_{ae}_twosides"]))

SignificanceResult(statistic=0.3291402943021916, pvalue=0.07799498841937424)


#### hyponatremia

In [21]:
ae = "hyponatremia"
dfci_data[f"proportion_{ae}"] = dfci_data[f"num_{ae}_cases"] / dfci_data["num_all_patients"]
dfci_test_twosides_scores = torch.load(BASE_DIR + f"temp/final_drug_regimen_ae_summary_dfci_relaxed_v2_{ae}_scores.pt")

dfci_data.loc[:, f"pred_{ae}_twosides"] = pd.DataFrame(dfci_test_twosides_scores[700][:, drug_1_ind_inds, drug_2_ind_inds], index=sum([outcome_mapper[ae]["twosides"] for ae in aes], start=[]), columns=list(zip(np.array(drug_names)[drug_1_ind_inds], np.array(drug_names)[drug_2_ind_inds]))).drop_duplicates().loc["Hyponatraemia"].values

plot_data = dfci_data.query("drug_name_1 not in @heme_drugs and drug_name_2 not in @heme_drugs and num_all_patients >= @detect_thres")
print(kendalltau(plot_data[f"proportion_{ae}"], plot_data[f"pred_{ae}_twosides"]))

SignificanceResult(statistic=0.6839855680567694, pvalue=0.0002792104157155115)


#### hypokalemia

In [22]:
ae = "hypokalemia"
dfci_data[f"proportion_{ae}"] = dfci_data[f"num_{ae}_cases"] / dfci_data["num_all_patients"]
dfci_test_twosides_scores = torch.load(BASE_DIR + f"temp/final_drug_regimen_ae_summary_dfci_relaxed_v2_{ae}_scores.pt")

dfci_data.loc[:, f"pred_{ae}_twosides"] = pd.DataFrame(dfci_test_twosides_scores[700][:, drug_1_ind_inds, drug_2_ind_inds], index=sum([outcome_mapper[ae]["twosides"] for ae in aes], start=[]), columns=list(zip(np.array(drug_names)[drug_1_ind_inds], np.array(drug_names)[drug_2_ind_inds]))).drop_duplicates().loc["Hypokalaemia"].values

plot_data = dfci_data.query("drug_name_1 not in @heme_drugs and drug_name_2 not in @heme_drugs and num_all_patients >= @detect_thres")
print(kendalltau(plot_data[f"proportion_{ae}"], plot_data[f"pred_{ae}_twosides"]))

SignificanceResult(statistic=0.3291402943021916, pvalue=0.07799498841937424)


#### hyperkalemia

In [23]:
ae = "hyperkalemia"
dfci_data[f"proportion_{ae}"] = dfci_data[f"num_{ae}_cases"] / dfci_data["num_all_patients"]
dfci_test_twosides_scores = torch.load(BASE_DIR + f"temp/final_drug_regimen_ae_summary_dfci_relaxed_v2_{ae}_scores.pt")

dfci_data.loc[:, f"pred_{ae}_twosides"] = pd.DataFrame(dfci_test_twosides_scores[700][:, drug_1_ind_inds, drug_2_ind_inds], index=sum([outcome_mapper[ae]["twosides"] for ae in aes], start=[]), columns=list(zip(np.array(drug_names)[drug_1_ind_inds], np.array(drug_names)[drug_2_ind_inds]))).drop_duplicates().loc["Hyperkalaemia"].values

plot_data = dfci_data.query("drug_name_1 not in @heme_drugs and drug_name_2 not in @heme_drugs and num_all_patients >= @detect_thres")
print(kendalltau(plot_data[f"proportion_{ae}"], plot_data[f"pred_{ae}_twosides"]))


SignificanceResult(statistic=0.5378528742004771, pvalue=0.007028071105256244)


#### hypomagnesemia

In [24]:
ae = "hypomagnesemia"
dfci_data[f"proportion_{ae}"] = dfci_data[f"num_{ae}_cases"] / dfci_data["num_all_patients"]
dfci_test_twosides_scores = torch.load(BASE_DIR + f"temp/final_drug_regimen_ae_summary_dfci_relaxed_v2_{ae}_scores.pt")
dfci_data.loc[:, f"pred_{ae}_twosides"] = pd.DataFrame(dfci_test_twosides_scores[700][:, drug_1_ind_inds, drug_2_ind_inds], index=sum([outcome_mapper[ae]["twosides"] for ae in aes], start=[]), columns=list(zip(np.array(drug_names)[drug_1_ind_inds], np.array(drug_names)[drug_2_ind_inds]))).drop_duplicates().loc["Hypomagnesaemia"].values

plot_data = dfci_data.query("drug_name_1 not in @heme_drugs and drug_name_2 not in @heme_drugs and num_all_patients >= @detect_thres")
print(kendalltau(plot_data[f"proportion_{ae}"], plot_data[f"pred_{ae}_twosides"]))

SignificanceResult(statistic=0.5256137757611014, pvalue=0.006217978477888536)


#### hypocalcemia

In [25]:
ae = "hypocalcemia"
dfci_data[f"proportion_{ae}"] = dfci_data[f"num_{ae}_cases"] / dfci_data["num_all_patients"]
dfci_test_twosides_scores = torch.load(BASE_DIR + f"temp/final_drug_regimen_ae_summary_dfci_relaxed_v2_{ae}_scores.pt")
dfci_data.loc[:, f"pred_{ae}_twosides"] = pd.DataFrame(dfci_test_twosides_scores[700][:, drug_1_ind_inds, drug_2_ind_inds], index=sum([outcome_mapper[ae]["twosides"] for ae in aes], start=[]), columns=list(zip(np.array(drug_names)[drug_1_ind_inds], np.array(drug_names)[drug_2_ind_inds]))).drop_duplicates().loc["Hypocalcaemia"].values

plot_data = dfci_data.query("drug_name_1 not in @heme_drugs and drug_name_2 not in @heme_drugs and num_all_patients >= @detect_thres")
print(kendalltau(plot_data[f"proportion_{ae}"], plot_data[f"pred_{ae}_twosides"]))

SignificanceResult(statistic=0.23777817717036512, pvalue=0.24647971342183916)


#### hypercalcemia

In [26]:
ae = "hypercalcemia"
dfci_data[f"proportion_{ae}"] = dfci_data[f"num_{ae}_cases"] / dfci_data["num_all_patients"]
dfci_test_twosides_scores = torch.load(BASE_DIR + f"temp/final_drug_regimen_ae_summary_dfci_relaxed_v2_{ae}_scores.pt")

dfci_data.loc[:, f"pred_{ae}_twosides"] = pd.DataFrame(dfci_test_twosides_scores[700][:, drug_1_ind_inds, drug_2_ind_inds], index=sum([outcome_mapper[ae]["twosides"] for ae in aes], start=[]), columns=list(zip(np.array(drug_names)[drug_1_ind_inds], np.array(drug_names)[drug_2_ind_inds]))).drop_duplicates().loc["Hypercalcaemia"].values

plot_data = dfci_data.query("drug_name_1 not in @heme_drugs and drug_name_2 not in @heme_drugs and num_all_patients >= @detect_thres")
print(kendalltau(plot_data[f"proportion_{ae}"], plot_data[f"pred_{ae}_twosides"]))

SignificanceResult(statistic=0.1102206622077278, pvalue=0.5824764678984073)


#### Summary

In [27]:
# Summary table: Kendall's tau correlation between observed AE incidence and Madrigal predictions
ae_display = {
    "neutropenia": "Neutropenia", "pancytopenia": "Pancytopenia", "anemia": "Anemia",
    "thrombocytopenia": "Thrombocytopenia", "polyneuropathy": "Polyneuropathy",
    "pulmonary_embolism/deep_vein_thrombosis": "Thromboembolism",
    "acute_kidney_injury": "Acute kidney injury", "hyponatremia": "Hyponatremia",
    "hypokalemia": "Hypokalemia", "hyperkalemia": "Hyperkalemia",
    "hypomagnesemia": "Hypomagnesemia", "hypocalcemia": "Hypocalcemia",
    "hypercalcemia": "Hypercalcemia",
}

def _fmt_p(p):
    if pd.isna(p):
        return "n/a"
    if p < 0.01:
        return "<0.01"
    if p > 0.20:
        return ">0.20"
    return f"{p:.2f}"

corr_rows = []
for ae in aes:
    plot_data = dfci_data.query("drug_name_1 not in @heme_drugs and drug_name_2 not in @heme_drugs and num_all_patients >= @detect_thres")
    tau, p = kendalltau(plot_data[f"proportion_{ae}"], plot_data[f"pred_{ae}_twosides"])
    corr_rows.append({
        "AE outcome": ae_display[ae],
        "Kendall's tau": round(tau, 2),
        "p-value": _fmt_p(p),
        "Max incidence": round(plot_data[f"proportion_{ae}"].max(), 2),
    })

kendall_summary_df = pd.DataFrame(corr_rows)
display(kendall_summary_df)
kendall_summary_df.to_csv("dfci_patient_drug_regimen_ae_correlation_summary.csv", index=False)

,AE outcome,Kendall's tau,p-value,Max incidence
0,Neutropenia,0.68,<0.01,0.14
1,Pancytopenia,0.45,0.02,0.12
2,Anemia,0.43,0.02,0.13
3,Thrombocytopenia,0.49,0.01,0.09
4,Polyneuropathy,0.44,0.02,0.05
5,Thromboembolism,0.54,<0.01,0.09
6,Acute kidney injury,0.33,0.08,0.06
7,Hyponatremia,0.68,<0.01,0.07
8,Hypokalemia,0.33,0.08,0.05
9,Hyperkalemia,0.54,<0.01,0.04


### Adjusting for patient characteristics

In [28]:
include_tissue_type = False  # Because of high collinearity between tumor type and drug combo
top_tissue_only = 10
standardize_scores = True 

In [29]:
dfci_patient_data_filtered = pd.read_pickle("./dfci/dfci_patient_data_filtered.pkl")
dfci_patient_drugs_twosides_scores = pd.read_pickle(BASE_DIR + "temp/dfci_patient_drugs_twosides_scores.pkl")

assert not dfci_patient_data_filtered.index.has_duplicates
dfci_patient_data_ml = dfci_patient_data_filtered.query("(drug_name_1 not in @heme_drugs and drug_name_2 not in @heme_drugs) and ICD_BASED_TISSUE_TYPE not in @heme_tissues")
dfci_patient_data_ml = dfci_patient_data_ml[~dfci_patient_data_ml["FIRST_DRUG_REGIMEN"].str.contains("PEGYLATEDLIPOSOMALDOXORUBICIN")]  # No match
dfci_patient_data_ml = dfci_patient_data_ml[dfci_patient_data_ml["FIRST_DRUG_REGIMEN"].isin(dfci_patient_data_ml["FIRST_DRUG_REGIMEN"].value_counts()[dfci_patient_data_ml["FIRST_DRUG_REGIMEN"].value_counts() >= detect_thres].index.values)]
dfci_patient_data_ml.loc[dfci_patient_data_ml["RACE"].isin(dfci_patient_data_ml["RACE"].value_counts()[dfci_patient_data_ml["RACE"].value_counts() < 100].index.values), "RACE"] = "OTHER"

# Keep only the 10 most frequent tissue types; everything else --> 'OTHER'
top_tissues = dfci_patient_data_ml["ICD_BASED_TISSUE_TYPE"].value_counts().nlargest(top_tissue_only).index
# Exclude UNSPECIFIED
if "UNSPECIFIED" in top_tissues:
    top_tissues = top_tissues.drop("UNSPECIFIED")
dfci_patient_data_ml["ICD_BASED_TISSUE_TYPE_REDUCED"] = np.where(
    dfci_patient_data_ml["ICD_BASED_TISSUE_TYPE"].isin(top_tissues),
    dfci_patient_data_ml["ICD_BASED_TISSUE_TYPE"],
    "Other",
)

In [30]:
if include_tissue_type:
    ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore", drop=["Other", "OTHER", "MALE"])
    dfci_patient_data_ml_patient_X = pd.DataFrame(ohe.fit_transform(dfci_patient_data_ml[["ICD_BASED_TISSUE_TYPE_REDUCED", "RACE", "GENDER_NM"]]), columns=ohe.get_feature_names_out(), index=dfci_patient_data_ml.index)
else:
    ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore", drop=["OTHER", "MALE"])
    dfci_patient_data_ml_patient_X = pd.DataFrame(ohe.fit_transform(dfci_patient_data_ml[["RACE", "GENDER_NM"]]), columns=ohe.get_feature_names_out(), index=dfci_patient_data_ml.index)
        
dfci_patient_data_ml_patient_X = pd.concat([
    dfci_patient_data_ml[["PALLIATIVE_INTENT", "AGE_AT_TREAT"]], 
    dfci_patient_data_ml_patient_X, 
], axis=1)

In [33]:
aes_score_inds = {
    "neutropenia": 0, 
    "pancytopenia": 0, 
    "anemia": 0,
    "thrombocytopenia": 0, 
    "polyneuropathy": 0,
    "pulmonary_embolism/deep_vein_thrombosis": 1, 
    "acute_kidney_injury": 0, 
    "hyponatremia": 0, 
    "hypokalemia": 0, 
    "hyperkalemia": 0, 
    "hypomagnesemia": 0, 
    "hypocalcemia": 0, 
    "hypercalcemia": 0, 
}

In [34]:
for ae in aes:
    print("\n"+ae)
    if os.path.exists(f"./dfci/dfci_patient_data_ml_{ae.replace('/', '_')}_logit_coef_table_{detect_thres}_{include_tissue_type}_{top_tissue_only}_{standardize_scores}.csv"):
        print(f"\nSkipping {ae} as it has already been processed.\n")
        continue

    dfci_patient_data_ml_drug_twosides_scores_X = dfci_patient_drugs_twosides_scores.drop_duplicates().T.drop_duplicates().T.loc[[o for o in outcome_mapper[ae]["twosides"] if o not in {"Adverse drug reaction", "Adverse event"}], dfci_patient_data_ml[["drug_name_1", "drug_name_2"]].apply(lambda row: (row["drug_name_1"], row["drug_name_2"]), axis=1).values].T
    dfci_patient_data_ml_drug_twosides_scores_X.columns = [col+"_twosides" for col in dfci_patient_data_ml_drug_twosides_scores_X.columns]
    dfci_patient_data_ml_drug_twosides_scores_X.index = dfci_patient_data_ml_patient_X.index

    # Prepare X and y
    X = [dfci_patient_data_ml_patient_X]
    
    score_ind = aes_score_inds[ae]
    X += [dfci_patient_data_ml_drug_twosides_scores_X.iloc[:, score_ind]]

    X = pd.concat(X, axis=1)
    y = dfci_patient_data_ml[f"AE_{ae}"].values

    def _standardise(df, scaler=None):
        """Z-score numeric (non-binary) cols; keep dummies as-is."""
        df = df.copy()
        num_mask = (df.dtypes == float) | (df.dtypes == int)
        num_cols = [c for c in df.columns[num_mask]
                    if set(df[c].dropna().unique()) - {0, 1}]
        if scaler is None:
            scaler = StandardScaler()
            df[num_cols] = scaler.fit_transform(df[num_cols])
            return df, scaler
        else:
            df[num_cols] = scaler.transform(df[num_cols])
            return df

    # -----------------------------------------------------------
    # 1.  Scale once on the full feature set
    # -----------------------------------------------------------
    if standardize_scores:  # If not float, _standardise will not standardize the scores, which are originally float32 or object
        X = X.astype(float)
        
    X_scaled, scaler = _standardise(X)
    X_np       = X_scaled.values.astype(float)
    y_arr      = y.copy()
    X_dup = X_np
    y_dup = y_arr
    X_dup_c = sm.add_constant(X_dup, has_constant='add')

    mod_full = sm.Logit(y_dup, X_dup_c)
    res_full = mod_full.fit(method="bfgs", maxiter=5_000, disp=False)

    params   = res_full.params                    # log-odds coefficients
    se       = res_full.bse
    pvalues  = res_full.pvalues
    ci_lo, ci_hi = res_full.conf_int().T   # log-odds CI limits

    # ---------- build table ----------
    coef_table = (
        pd.DataFrame({
            "feature"   : ["const"] + X_scaled.columns.tolist(),            # includes 'const'
            "beta"      : params,           # log-odds
            "OR"        : np.exp(params),   # odds ratio
            "CI_low"    : np.exp(ci_lo),           # 95 % CI lower bound (OR scale)
            "CI_high"   : np.exp(ci_hi),           # 95 % CI upper bound
            "se"        : se,
            "p"         : pvalues,
        })
        .sort_values("p")                          # smallest p on top
        .reset_index(drop=True)
    )
    
    coef_table.to_csv(f"./dfci/dfci_patient_data_ml_{ae.replace('/', '_')}_logit_coef_table_{detect_thres}_{include_tissue_type}_{top_tissue_only}_{standardize_scores}.csv", index=False)



neutropenia

pancytopenia

anemia

thrombocytopenia

polyneuropathy

pulmonary_embolism/deep_vein_thrombosis

acute_kidney_injury

hyponatremia

hypokalemia

hyperkalemia

hypomagnesemia

hypocalcemia

hypercalcemia


In [35]:
# Summary table: logistic regression log-odds of Madrigal scores, controlled for patient characteristics
ae_display = {
    "neutropenia": "Neutropenia", "pancytopenia": "Pancytopenia", "anemia": "Anemia",
    "thrombocytopenia": "Thrombocytopenia", "polyneuropathy": "Polyneuropathy",
    "pulmonary_embolism/deep_vein_thrombosis": "Thromboembolism",
    "acute_kidney_injury": "Acute kidney injury", "hyponatremia": "Hyponatremia",
    "hypokalemia": "Hypokalemia", "hyperkalemia": "Hyperkalemia",
    "hypomagnesemia": "Hypomagnesemia", "hypocalcemia": "Hypocalcemia",
    "hypercalcemia": "Hypercalcemia",
}

def _fmt_p(p):
    if pd.isna(p):
        return "n/a"
    if p < 0.01:
        return "<0.01"
    if p > 0.20:
        return ">0.20"
    return f"{p:.2f}"

logodds_rows = []
for ae in aes:
    fname = f"./dfci/dfci_patient_data_ml_{ae.replace('/', '_')}_logit_coef_table_{detect_thres}_{include_tissue_type}_{top_tissue_only}_{standardize_scores}.csv"
    coef_table = pd.read_csv(fname)
    row = coef_table[coef_table["feature"].astype(str).str.endswith("_twosides")].iloc[0]
    logodds_rows.append({
        "AE outcome": ae_display[ae],
        "Log odds": round(row["beta"], 2),
        "SE": round(row["se"], 2),
        "p-value": _fmt_p(row["p"]),
    })

logodds_summary_df = pd.DataFrame(logodds_rows)
display(logodds_summary_df)
logodds_summary_df.to_csv("dfci_patient_drug_regimen_ae_logodds_summary_logit.csv", index=False)

,AE outcome,Log odds,SE,p-value
0,Neutropenia,0.65,0.15,<0.01
1,Pancytopenia,0.16,0.16,>0.20
2,Anemia,0.46,0.12,<0.01
3,Thrombocytopenia,0.63,0.18,<0.01
4,Polyneuropathy,1.41,0.44,<0.01
5,Thromboembolism,0.57,0.15,<0.01
6,Acute kidney injury,0.49,0.19,<0.01
7,Hyponatremia,0.79,0.21,<0.01
8,Hypokalemia,0.31,0.17,0.07
9,Hyperkalemia,1.12,0.48,0.02


### Patient-level cohort

In [36]:
include_tissue_type = False  # tumour tissue type is near-collinear with the two-drug regimen; excluded
top_tissue_only = 10

include_one_hot_drug_pair = False
standardize_scores = False

dfci_patient_data_filtered = pd.read_pickle("./dfci/dfci_patient_data_filtered.pkl")

assert not dfci_patient_data_filtered.index.has_duplicates
dfci_patient_data_ml = dfci_patient_data_filtered.query("(drug_name_1 not in @heme_drugs and drug_name_2 not in @heme_drugs) and ICD_BASED_TISSUE_TYPE not in @heme_tissues")
dfci_patient_data_ml = dfci_patient_data_ml[~dfci_patient_data_ml["FIRST_DRUG_REGIMEN"].str.contains("PEGYLATEDLIPOSOMALDOXORUBICIN")]  # no match
dfci_patient_data_ml.loc[dfci_patient_data_ml["RACE"].isin(dfci_patient_data_ml["RACE"].value_counts()[dfci_patient_data_ml["RACE"].value_counts() < 100].index.values), "RACE"] = "OTHER"

# Keep only the 10 most frequent tissue types; everything else --> 'OTHER'
top5_tissues = dfci_patient_data_ml["ICD_BASED_TISSUE_TYPE"].value_counts().nlargest(top_tissue_only).index
# Exclude UNSPECIFIED
if "UNSPECIFIED" in top5_tissues:
    top5_tissues = top5_tissues.drop("UNSPECIFIED")
dfci_patient_data_ml["ICD_BASED_TISSUE_TYPE_REDUCED"] = np.where(
    dfci_patient_data_ml["ICD_BASED_TISSUE_TYPE"].isin(top5_tissues),
    dfci_patient_data_ml["ICD_BASED_TISSUE_TYPE"],
    "Other",
)

drop_cols = ["MALE"]
patient_data_cols = ["GENDER_NM"]
if include_tissue_type:
    drop_cols += ["Other"]
    patient_data_cols += ["ICD_BASED_TISSUE_TYPE_REDUCED"]
drop_cols += ["OTHER"]
patient_data_cols += ["RACE"]
        
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore", drop=drop_cols)
dfci_patient_data_ml_patient_X = pd.DataFrame(ohe.fit_transform(dfci_patient_data_ml[patient_data_cols]), columns=ohe.get_feature_names_out(), index=dfci_patient_data_ml.index)
dfci_patient_data_ml_patient_X = pd.concat([
    dfci_patient_data_ml[["PALLIATIVE_INTENT", "AGE_AT_TREAT"]],
    dfci_patient_data_ml_patient_X
], axis=1)

print(dfci_patient_data_ml.shape, dfci_patient_data_ml_patient_X.shape)

(3577, 26) (3577, 15)


In [38]:
dfci_patient_data_ml["FIRST_DRUG_REGIMEN"].nunique()

26